### Create the enriched view
Joins fact_orders with all dimension tables into one flat view for the dashboard, with revenue = quantity × yearly price.

In [0]:
CREATE OR REPLACE VIEW fmcg.gold.view_fact_orders_enriched AS
SELECT
    fo.date,
    fo.product_code,
    fo.customer_code,

    -- Company
    CASE WHEN dc.platform = 'Sports Bar' THEN 'SportsBar' ELSE 'Atlikon' END AS company,

    -- Date attributes
    dd.date_key,
    dd.year,
    dd.month_name,
    dd.month_short_name,
    dd.quarter,
    dd.year_quarter,

    -- Customer attributes
    dc.customer,
    dc.market,
    dc.platform,
    dc.channel,

    -- Product attributes
    dp.division,
    dp.category,
    dp.product,
    dp.variant,

    -- Metrics
    fo.sold_quantity,
    gp.price_inr,
    fo.sold_quantity * gp.price_inr AS total_amount_inr

FROM fmcg.gold.fact_orders fo
LEFT JOIN fmcg.gold.dim_date dd
       ON fo.date = dd.month_start_date
LEFT JOIN fmcg.gold.dim_customers dc
       ON fo.customer_code = dc.customer_code
LEFT JOIN fmcg.gold.dim_products dp
       ON fo.product_code = dp.product_code
LEFT JOIN fmcg.gold.dim_gross_price gp
       ON fo.product_code = gp.product_code
      AND YEAR(fo.date) = gp.year;

### Check: no rows lost or duplicated
The view must have exactly as many rows as fact_orders (101,212).

In [0]:
SELECT COUNT(*) AS view_rows FROM fmcg.gold.view_fact_orders_enriched;

view_rows
101212


### Check: missing matches
Counts rows where a join found nothing. Should all be 0.

In [0]:
SELECT
    SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END)      AS missing_date,
    SUM(CASE WHEN customer IS NULL THEN 1 ELSE 0 END)  AS missing_customer,
    SUM(CASE WHEN product IS NULL THEN 1 ELSE 0 END)   AS missing_product,
    SUM(CASE WHEN price_inr IS NULL THEN 1 ELSE 0 END) AS missing_price
FROM fmcg.gold.view_fact_orders_enriched;

missing_date,missing_customer,missing_product,missing_price
0,0,0,0


### Preview

In [0]:
SELECT * FROM fmcg.gold.view_fact_orders_enriched LIMIT 20;

date,product_code,customer_code,company,date_key,year,month_name,month_short_name,quarter,year_quarter,customer,market,platform,channel,division,category,product,variant,sold_quantity,price_inr,total_amount_inr
2025-12-01,ARCHAFF0E4,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Bow,AX Precision Recurve Bow,26 lbs,930,4990,4640700
2025-12-01,ARCH6B94F7,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Bow,AX Precision Recurve Bow,30 lbs,878,4060,3564680
2025-12-01,ARCH5D1FE7,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Bow Stringer,BX Bow Stringing Tool,Standard,1089,1760,1916640
2025-12-01,ARCH7B49A9,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Arm Guard,NX Archery Arm Guard,Medium,1385,1720,2382200
2025-12-01,ARCH497D34,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Target,TX Foam Archery Target,80 cm,1063,1200,1275600
2025-12-01,ARCHE71D79,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Arm Guard,NX Archery Arm Guard,Large,704,6140,4322560
2025-12-01,ARCHDD8749,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Archery,Target,TX Foam Archery Target,60 cm,1708,4290,7327320
2025-12-01,BADMC045D4,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Badminton,Badminton Racket,BX AeroLite Badminton Racket,Head Heavy,1345,4760,6402200
2025-12-01,BADM4BAB38,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Badminton,Badminton Racket,BX AeroLite Badminton Racket,Advanced,347,6700,2324900
2025-12-01,BADMDC6AB1,70002018,Atlikon,202512,2025,December,Dec,Q4,2025-Q4,PlayMore Sports,India,Brick & Mortar,Retailer,Badminton,Badminton Racket,AX PowerDrive Badminton Racket,Even Balance,319,1920,612480
